# 4D SfM — DEM + DoD + M3C2 raster (monthly batch)

Runs the full per-date workflow for one date per month, producing
DEM + orthoimage + DoD + stable-terrain DoD + M3C2 raster (with
histograms) for each date. All logic lives in
`cntp.pipeline_4dsfm.run_4dsfm_day_with_rasters`; this notebook is
just configuration + one loop.

Reference rasters (`reference_dem.tif`, `reference_ortho.tif`,
`reference_dem_stable.tif`) cache in `_ref_cache/` — built on the
first iteration, skipped on every subsequent date.

In [2]:
%load_ext autoreload
%autoreload 2

import os
os.environ["AGISOFT_LICENSE_PATH"] = "/home/asus/.config/Agisoft/license.lic"

from pathlib import Path

import Metashape  # noqa: F401  — must import after AGISOFT_LICENSE_PATH is set
from cntp.pipeline_4dsfm import run_4dsfm_day_with_rasters

## Configuration

Edit only this section. Paths + dates + all per-stage knobs live here.

In [3]:
# ── Site — edit the 3 paths in site_config.py to choose / switch glacier ─
import site_config as site

# ── Dates to process (one per month) ─────────────────────────────────
monthly_dates = [
    "2024-01-18",
    "2024-02-18",
    "2024-03-17",
    "2024-04-19",
    "2024-05-17",
    "2024-06-15",
    "2024-07-15",
    "2024-08-17",
    "2024-09-18",
    "2024-10-16",
    "2024-11-15",
]

# ── Per-date knobs (SfM + raster combined) ───────────────────────────
params = dict(
    # SfM pipeline knobs (forwarded to run_4dsfm_day)
    match_downscale       = 1,
    depth_downscale       = 2,
    filter_mode           = "Aggressive",        # "Mild" or "Aggressive"
    loc_acc_new           = (0.5, 0.5, 0.5),
    rot_acc_new           = (5.0, 5.0, 5.0),
    ref_downsample        = 0.4,   # per-glacier coreg knob (0.05 North dense / 0.40 West)
    tba_downsample        = 1.0,
    p2p_max_disp          = 10,
    sp2p_max_disp         = 5,
    m_sp2p_max_disp       = 0.1,
    use_ecef              = True,
    overwrite             = False,
    verbose               = True,
    # Registry frozen at the 2023-11-27 baseline (no feedback into BA).
    add_to_registry       = False,
    # Skip Step 6 rebuild + Step 6b validation (coreg M3C2 plot still runs).
    run_validation        = False,

    # Raster knobs
    dem_method            = "point2dem",   # HSfM ASP point2dem (IDW); "cubic" = legacy
    res                   = 1.0,
    max_gap_pixels        = 1,
    ref_cloud_downsample  = 0.25,
    m3c2_ref_downsample   = 0.25,
    slope_threshold       = 60.0,
    overwrite_ref_dem     = False,
    overwrite_day_dem     = False,
    overwrite_dod         = False,
    overwrite_stable      = False,
    overwrite_stable_dod  = False,
    overwrite_m3c2        = False,
)

## Run

Loops over `monthly_dates` and calls `run_4dsfm_day_with_rasters`
for each. Wrapped in `try / except` so a single bad date doesn't
stop the run; per-date stats collected and printed at the end.

In [6]:
import traceback

summary = []
for d in monthly_dates:
    try:
        summary.append(run_4dsfm_day_with_rasters(
            new_date     = d,
            tlcam_dir    = site.tlcam_dir,
            ref_cloud    = site.ref_cloud,
            glacier_mask = site.glacier_mask,
            registry_csv = site.registry_csv,
            output_dir   = site.output_dir,
            **params,
        ))
    except Exception as e:
        print(f"\n  {d} failed: {type(e).__name__}: {e}")
        traceback.print_exc()
        summary.append({"date": d, "error": repr(e)})

print(f"\n{'='*70}\n  Batch summary ({len(summary)} dates)\n{'='*70}")
for s in summary:
    if "error" in s:
        print(f"  {s['date']} : ERROR — {s['error']}")
    else:
        print(
            f"  {s['date']} : "
            f"DoD med={s['dod_stats']['median']:+.3f} m  std={s['dod_stats']['std']:.3f}  |  "
            f"stable med={s['stable_stats']['median']:+.3f} m  std={s['stable_stats']['std']:.3f}  |  "
            f"M3C2 med={s['m3c2_stats']['median']:+.3f} m  std={s['m3c2_stats']['std']:.3f}"
        )

[Step 1] Skipping — 2024-01-18_cameras_4DSfM.csv exists
[Step 2] Skipping — 2024-01-18_cloud.las exists
  Stable reference cached → Reference_UAV_TLC_PCS_ds0.40_stable.las
[Step 3] Skipping — 2024-01-18_cloud_coreg_hsfm.las exists
[Step 3b] Skipping — stable TBA exists
[Step 4] Skipping — 2024-01-18_cameras_coreg.csv exists

[Step 6] Skipping — run_validation=False
[Step 6b] Skipping — run_validation=False

[Step 7] Skipping — add_to_registry=False (registry kept frozen)
  DEM + ortho cached → reference_dem.tif, reference_ortho.tif
  DEM + ortho cached → 2024-01-18_dem.tif, 2024-01-18_ortho.tif
  DoD cached → DOD.tif
  Stable DEM cached → reference_dem_stable.tif
  Stable DEM cached → 2024-01-18_dem_stable.tif
  DoD cached → DOD_stable.tif
  M3C2 raster cached → M3C2_raster.tif

  [2024-01-18] DoD med=+0.375 m  |  stable med=-0.470 m  |  M3C2 med=+0.272 m
[Step 1] Skipping — 2024-02-18_cameras_4DSfM.csv exists
[Step 2] Skipping — 2024-02-18_cloud.las exists
  Stable reference cached → 